In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from docling.document_converter import DocumentConverter


def norm_list(xs):
    return [str(x).replace("\u00a0", " ").strip() for x in xs]


def header_similarity(a, b):
    a = norm_list(a)
    b = norm_list(b)
    n = max(1, min(len(a), len(b)))
    return sum(a[i] == b[i] for i in range(n)) / n


def looks_like_data(cols):
    s = " ".join(norm_list(cols))
    return bool(re.search(r"\d", s)) or ("%" in s)


# minden oldal elejen leveszi a headert ahol nem kell oda
def fix_df(df, global_header):
    df = df.copy()
    cols = norm_list(df.columns.tolist())
    gh = norm_list(global_header)

    if header_similarity(cols, gh) >= 0.9:
        df.columns = gh
        return df

    # ha az oszlopnevek inkabb adatnak neznek ki, mentsuk vissza oket elso sornak.
    if looks_like_data(cols):
        row_df = pd.DataFrame([cols], columns=gh[: len(cols)])
        df.columns = gh[: len(df.columns)]
        out = pd.concat([row_df, df], ignore_index=True)
        out.columns = gh[: len(out.columns)]
        return out

    df.columns = gh[: len(df.columns)]
    return df


# a cegnevet es a cimet szetszedi
def split_name_address(text):
    if pd.isna(text):
        return pd.NA, pd.NA

    s = str(text).strip()
    # 4 számjegy + szóköz + 1 szó + vessző
    m = re.search(r"\b(\d{4}\s+\S+,\s*.*)", s)
    if m:
        address = m.group(1).strip()
        name = s[: m.start()].strip()
        return name, address

    return s, pd.NA


# arbevetelt adatait formazza
def hu_to_float(x):
    if x is None or pd.isna(x):
        return pd.NA

    s = str(x).strip()
    if s in ["N/A", "NA", "-", ""]:
        return pd.NA

    s = s.replace(" ", "").replace("\u00a0", "")
    s = s.replace(".", "")
    s = s.replace(",", ".")
    return s


# kiveszi a nevet a tablazat elott
def extract_name_from_doc(doc):
    md = doc.document.export_to_markdown()
    for line in md.splitlines():
        line = line.strip()
        if line.startswith("##"):
            return re.sub(r"^#+", "", line).strip()
    return pd.NA

In [2]:
def pdf_to_dataframe(pdf_path: Path, converter: DocumentConverter) -> pd.DataFrame:
    doc = converter.convert(str(pdf_path))

    dfs = [tbl.export_to_dataframe(doc.document) for tbl in doc.document.tables]
    if not dfs:
        return pd.DataFrame()

    global_header = norm_list(dfs[0].columns.tolist())
    fixed = [fix_df(d, global_header) for d in dfs]

    big = pd.concat(fixed, ignore_index=True)

    # üres sorok dobasa
    big = big.dropna(how="all")
    big = big[(big.astype(str).apply(lambda r: "".join(r).strip(), axis=1) != "")]

    # nev kinyerese és beszurasa
    person_name = extract_name_from_doc(doc)
    big.insert(0, "Név", person_name)

    # Érdekeltség -> Cégnév, Cím
    if "Érdekeltség" in big.columns:
        big[["Cégnév", "Cím"]] = big["Érdekeltség"].apply(
            lambda x: pd.Series(split_name_address(x))
        )
        big = big.drop(columns=["Érdekeltség"])

    if "Árbevétel (2024) (eFt)" in big.columns:
        big["Árbevétel (2024) (eFt)"] = pd.to_numeric(
            big["Árbevétel (2024) (eFt)"].map(hu_to_float), errors="coerce"
        )

    if "Tulajdonosi lánc hossza 2" in big.columns:
        big["Tulajdonosi lánc hossza 2"] = pd.to_numeric(
            big["Tulajdonosi lánc hossza 2"], errors="coerce"
        )

    if "Befolyás mértéke 1" in big.columns:
        big["Befolyás mértéke 1"] = pd.to_numeric(
            big["Befolyás mértéke 1"]
            .astype("string")
            .str.replace("%", "", regex=False)
            .str.strip()
            .replace("", pd.NA),
            errors="coerce",
        )

    return big


def batch_pdf_to_excel(input_dir: str | Path, output_dir: str | Path) -> list[Path]:
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # környezeti beállítások
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
    os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
    os.environ["HF_HOME"] = str(Path.home() / "hf_cache")
    os.environ["TRANSFORMERS_CACHE"] = str(Path.home() / "hf_cache")

    converter = DocumentConverter()

    pdf_files = sorted(input_dir.glob("*.pdf"))
    written = []

    for pdf_path in pdf_files:
        try:
            df = pdf_to_dataframe(pdf_path, converter)

            out_path = output_dir / f"{pdf_path.stem}.xlsx"
            with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
                df.to_excel(writer, index=False, sheet_name="Adatok")

            written.append(out_path)
            print(f"[OK] {pdf_path.name} -> {out_path.name}  (sorok: {len(df)})")

        except Exception as e:
            print(f"[HIBA] {pdf_path.name}: {e}")

    return written


batch_pdf_to_excel("input_pdf_02", "output_excel")

[INFO] 2026-02-18 07:06:15,744 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-18 07:06:15,788 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-18 07:06:15,795 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.6.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-18 07:06:17,626 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-02-18 07:06:19,467 [RapidOCR] download_file.py:95: Successfully saved to: C:\Users\savoly\AppData\Local\pypoetry\Cache\virtualenvs\amo-playground-Tj7rmwcV-py3.13\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-18 07:06:19,471 [RapidOCR] main.py:50: Using C:\Users\savoly\AppData\Local\pypoetry\Cache\virtualenvs\amo-playground-Tj7rmwcV-py3.13\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-18 07:06:19,888 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-18 07:06:19,890 [RapidOCR

[OK] Antal László_eid14402024.pdf -> Antal László_eid14402024.xlsx  (sorok: 9)


[INFO] 2026-02-18 07:06:43,696 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-18 07:06:43,697 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-18 07:06:43,700 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\savoly\AppData\Local\pypoetry\Cache\virtualenvs\amo-playground-Tj7rmwcV-py3.13\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-02-18 07:06:43,701 [RapidOCR] main.py:50: Using C:\Users\savoly\AppData\Local\pypoetry\Cache\virtualenvs\amo-playground-Tj7rmwcV-py3.13\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-02-18 07:06:43,776 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-18 07:06:43,777 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-02-18 07:06:43,803 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\savoly\AppData\Local\pypoetry\Cache\virtualenvs\amo-playground-Tj7rmwcV-py3.13\Lib\site-packages\rapidocr\models\ch_PP-OC

[OK] Bene Gábor Zsigmond_eid15582670.pdf -> Bene Gábor Zsigmond_eid15582670.xlsx  (sorok: 10)
[OK] Bige László Tibor_eid15939751.pdf -> Bige László Tibor_eid15939751.xlsx  (sorok: 19)
[OK] Bige Zalán László_eid15889941.pdf -> Bige Zalán László_eid15889941.xlsx  (sorok: 8)
[OK] Bige Zoltán József_eid15936724.pdf -> Bige Zoltán József_eid15936724.xlsx  (sorok: 22)
[OK] Cséri László Gábor_eid15075582.pdf -> Cséri László Gábor_eid15075582.xlsx  (sorok: 4)
[OK] dr. Hütter Csaba László_eid14594008.pdf -> dr. Hütter Csaba László_eid14594008.xlsx  (sorok: 3)
[OK] Dr. Kaliczka Nándor_eid17463610.pdf -> Dr. Kaliczka Nándor_eid17463610.xlsx  (sorok: 3)
[OK] dr. Szatmári Melinda_eid14794290.pdf -> dr. Szatmári Melinda_eid14794290.xlsx  (sorok: 29)
[OK] Fekete Mihály_eid15144758.pdf -> Fekete Mihály_eid15144758.xlsx  (sorok: 2)
[OK] Fuchs Zsolt_eid15939064.pdf -> Fuchs Zsolt_eid15939064.xlsx  (sorok: 3)
[OK] György Tamás Norbert_eid15580344.pdf -> György Tamás Norbert_eid15580344.xlsx  (sorok: 42)


[WindowsPath('output_excel/Antal László_eid14402024.xlsx'),
 WindowsPath('output_excel/Bene Gábor Zsigmond_eid15582670.xlsx'),
 WindowsPath('output_excel/Bige László Tibor_eid15939751.xlsx'),
 WindowsPath('output_excel/Bige Zalán László_eid15889941.xlsx'),
 WindowsPath('output_excel/Bige Zoltán József_eid15936724.xlsx'),
 WindowsPath('output_excel/Cséri László Gábor_eid15075582.xlsx'),
 WindowsPath('output_excel/dr. Hütter Csaba László_eid14594008.xlsx'),
 WindowsPath('output_excel/Dr. Kaliczka Nándor_eid17463610.xlsx'),
 WindowsPath('output_excel/dr. Szatmári Melinda_eid14794290.xlsx'),
 WindowsPath('output_excel/Fekete Mihály_eid15144758.xlsx'),
 WindowsPath('output_excel/Fuchs Zsolt_eid15939064.xlsx'),
 WindowsPath('output_excel/György Tamás Norbert_eid15580344.xlsx'),
 WindowsPath('output_excel/Harsányi Havaska_eid17934196.xlsx'),
 WindowsPath('output_excel/Harsányi Virág_eid17519255.xlsx'),
 WindowsPath('output_excel/Harsányi Vivien_eid15768653.xlsx'),
 WindowsPath('output_excel/H

In [3]:
data = pd.concat(
    [
        pd.read_excel(file, engine="calamine")
        for file in sorted(Path("output_excel").glob("*.xlsx"))
    ]
)

In [4]:
data.to_excel("merged_entities_02.xlsx", index=False)